# Dioptra-DINO: Resume Fine-Tuning from Checkpoint Step Latest
### Resuming from Global Step 130,110 (Epoch 2) with 3D VNL + Wide Pinhole on TartanAir Indoor Suite

- **Checkpoint**: `checkpoint_step_latest.pt` (global_step=130110, epoch=2)
- **Backbone**: DINOv2 ViT-Small/14 with Trivision Ray PE (108 dims)
- **Supervision**: SiLog + Scale Calibration + Edge-Aware + 3D Virtual Normal Loss
- **Compute**: Dual Tesla T4 GPUs via nn.DataParallel


In [ ]:
# [1] GPU Detection & Environment Setup
import os, sys, shutil, glob
import torch

print("=" * 70)
print("DIOPTRA-DINO FINE-TUNE RESUME: HARDWARE & RUNTIME SETUP")
print("=" * 70)

if torch.cuda.is_available():
    gpu_count = torch.cuda.device_count()
    gpu_name = torch.cuda.get_device_name(0)
    major, minor = torch.cuda.get_device_capability(0)
    print(f"Detected {gpu_count}x GPU(s): {gpu_name} (Compute Capability sm_{major}{minor})")
    
    # If allocated Pascal GPU (sm_60 / Tesla P100), auto-downgrade to PyTorch 2.4.1 for sm_60 support
    if major < 7:
        print(f"Pascal GPU (sm_{major}{minor}) detected. PyTorch 2.6 requires sm_70+.")
        print("Installing PyTorch 2.4.1+cu121 with native sm_60 binary support...")
        os.system("pip install -q --no-cache-dir torch==2.4.1 torchvision==0.19.1 --index-url https://download.pytorch.org/whl/cu121")
        print("PyTorch reinstallation complete!")
    else:
        print(f"Modern GPU ({gpu_name}, sm_{major}{minor}). Native CUDA kernels ready.")
        if gpu_count > 1:
            print(f"Dual GPU detected! nn.DataParallel will distribute across all {gpu_count} GPUs.")
else:
    print("WARNING: No CUDA GPU detected.")

# Sync Dioptra-DINO code from attached input datasets (tesseract-v17)
code_roots = glob.glob('/kaggle/input/**/dioptra_dino.py', recursive=True)
if code_roots:
    src_dir = os.path.dirname(code_roots[0])
    for fname in ['dioptra_dino.py', 'dioptra.py']:
        src = os.path.join(src_dir, fname)
        dst = os.path.join('/kaggle/working', fname)
        if os.path.exists(src):
            shutil.copyfile(src, dst)
            print(f'Synchronized {fname} from {src_dir} -> /kaggle/working/')

# Also try GitHub clone for latest code
repo_dir = '/kaggle/working/dioptra_repo'
try:
    if os.path.exists(os.path.join(repo_dir, '.git')):
        os.system(f'cd {repo_dir} && git reset --hard && git pull origin main')
    else:
        os.system(f'git clone https://github.com/SeranomTheGreat/dioptra.git {repo_dir}')
    for fname in ['dioptra_dino.py', 'dioptra.py']:
        src = os.path.join(repo_dir, fname)
        dst = os.path.join('/kaggle/working', fname)
        if os.path.exists(src):
            shutil.copyfile(src, dst)
            print(f'Synchronized {fname} from GitHub -> /kaggle/working/')
except Exception as e:
    print(f'GitHub sync note: {e}')

for p in ['/kaggle/working', repo_dir]:
    if p not in sys.path:
        sys.path.insert(0, p)

print('>>> Runtime setup complete! <<<')


In [ ]:
# [2] Locate Checkpoint & Dataset
import os, sys, glob
from pathlib import Path

if '/kaggle/working' not in sys.path:
    sys.path.insert(0, '/kaggle/working')

print("=" * 70)
print("LOCATING CHECKPOINT & DATASET")
print("=" * 70)

# Find checkpoint
ckpt_path = None
ckpt_candidates = (
    glob.glob('/kaggle/input/**/checkpoint_step_latest.pt', recursive=True) +
    glob.glob('/kaggle/input/**/checkpoint_latest.pt', recursive=True) +
    glob.glob('/kaggle/input/**/dioptra_dino_epoch_*.pt', recursive=True) +
    glob.glob('/kaggle/input/**/dioptra_dino_best.pt', recursive=True)
)

if ckpt_candidates:
    # Prefer step_latest > latest > epoch > best
    for pattern in ['checkpoint_step_latest', 'checkpoint_latest', 'epoch_', 'best']:
        for c in ckpt_candidates:
            if pattern in os.path.basename(c):
                ckpt_path = c
                break
        if ckpt_path:
            break
    if not ckpt_path:
        ckpt_path = ckpt_candidates[0]

if ckpt_path:
    ckpt_mb = os.path.getsize(ckpt_path) / (1024 * 1024)
    print(f'>>> Found checkpoint: {ckpt_path} ({ckpt_mb:.1f} MB) <<<')
    
    # Copy to outputs_dino for seamless resume
    os.makedirs('/kaggle/working/outputs_dino', exist_ok=True)
    resume_dst = '/kaggle/working/outputs_dino/checkpoint_step_latest.pt'
    if not os.path.exists(resume_dst):
        import shutil
        shutil.copyfile(ckpt_path, resume_dst)
        print(f'Copied checkpoint to {resume_dst}')
else:
    print('WARNING: No checkpoint found in /kaggle/input/!')
    print('Training will start from scratch.')

# Find TartanAir dataset
wh_roots = sorted(
    glob.glob('/kaggle/input/**/tartanair-warehouse-stereo-suite/**/warehouse_stereo', recursive=True) +
    glob.glob('/kaggle/input/**/warehouse_stereo', recursive=True) +
    glob.glob('/kaggle/input/**/carwelding', recursive=True) +
    glob.glob('/kaggle/input/**/abandonedfactory', recursive=True)
)

data_path = None
if wh_roots:
    for cand in wh_roots:
        p = Path(cand)
        if p.name in ('carwelding', 'abandonedfactory'):
            data_path = str(p.parent)
            break
        elif (p / 'carwelding').exists() or (p / 'abandonedfactory').exists():
            data_path = str(p)
            break
    if not data_path:
        data_path = wh_roots[0]

if not data_path:
    try:
        from dioptra_dino import resolve_dataset_root
        data_path = resolve_dataset_root('auto')
    except:
        data_path = 'auto'

os.environ['DATA_PATH'] = data_path
print(f'>>> Dataset root: {data_path} <<<')

# Count frames
pngs = glob.glob(os.path.join(data_path, '**', '*.png'), recursive=True)
npys = glob.glob(os.path.join(data_path, '**', '*.npy'), recursive=True)
print(f'Found {len(pngs)} RGB frames and {len(npys)} GT depth maps.')


In [ ]:
# [3] Launch Fine-Tuning from Checkpoint
# Resume training from checkpoint_step_latest.pt (global step 130110, epoch 2)
# Training continues for 40 epochs with full optimizer/scheduler state restored
import os

data_path = os.environ.get('DATA_PATH', 'auto')
cmd = (
    f'python /kaggle/working/dioptra_dino.py '
    f'--train "{data_path}" '
    f'--epochs 40 '
    f'--batch-size 8 '
    f'--weight-normal 0.25 '
    f'--crop-min 0.35 '
    f'--lr-backbone 2e-5 '
    f'--lr-head 2e-4 '
    f'--resume auto'
)
print('='*70)
print('LAUNCHING FINE-TUNE TRAINING')
print('='*70)
print(f'Command: {cmd}')
print()
exit_code = os.system(cmd)
print(f'\nTraining finished with exit code: {exit_code}')
assert exit_code == 0, f'Training failed with exit code {exit_code}'


In [ ]:
# [4] Verify Output Checkpoints & Package Results
import os, glob
from IPython.display import FileLink

output_dir = '/kaggle/working/outputs_dino'
if os.path.exists(output_dir):
    print(f'Checkpoints in {output_dir}:')
    for f in sorted(os.listdir(output_dir)):
        p = os.path.join(output_dir, f)
        if os.path.isfile(p):
            size_mb = os.path.getsize(p) / (1024 * 1024)
            print(f'  {f} ({size_mb:.2f} MB)')
else:
    print('outputs_dino directory not yet created.')

# Package all outputs for download
os.system('cd /kaggle/working && zip -q -r dioptra_dino_finetune_results.zip outputs_dino/')
print('\n' + '='*70)
print('>>> SUCCESS! Fine-tuned checkpoints packaged! <<<')
print('Download: /kaggle/working/dioptra_dino_finetune_results.zip')
print('='*70)
display(FileLink('dioptra_dino_finetune_results.zip'))
